# EDA on silver layer data to identify exactly what should be cleaned

In [0]:
%python
from pyspark.sql.functions import (
    col,
    when,
    trim,
    regexp_replace,
    dense_rank,
    sum as _sum,
    right,
    to_date,
    lit,
    expr,
)
from pyspark.sql.window import Window

In [0]:
df_bronze = spark.sql("SELECT * FROM marathos.bronze.raw_marathon_data")

In [0]:
df_bronze.printSchema()
df_bronze.display()

## Distance format
- What kind of different event distances are there in this dataset and how do they differ?
    - There are over 2000 ways of writing the DISTANCE of the marathon which is crazy.

In [0]:
df_bronze.groupBy("event_distance_length").count().orderBy(col("count").desc()).display()

## How many event names does NOT end with a country code inside of parenthesis? 
- Example: Ishigaki Island Ultra Marathon (JPN)
    - What the cell proves is that it is alot of cleaning that is needed in this data. Output was:


| event_name | 
| :--- | 
| """6hs Solidarias """"Corré por los Chicos"""" (ARG)""" |
| """The """"James"""" Stampede 50 (NZL)""" |
| """Backyard Ultra """"THE LAST LAP"""" (SUI)""" |
| """24hs Solidarias """"Corré por los Chicos"""" (ARG)""" |
| """Tortugas """"A"""" Mountain 24 Hour Challenge (USA)""" |
| """12hs Solidarias """"Corré por los Chicos"""" (ARG)""" |
| """Ultra Trail Race """"El Chico Hidalgo"""" (MEX)""" |
| """6 Horas individual y posta """"Aniversario Ciudad de Bahía Blanca"""" (ARG)""" |
| """6 ore al Parco Adda-Mallero """"Renato Bartesaghi"""" (ITA)""" |
| """The """"James"""" Stampede 45 (NZL)""" |


- There is alot of garbage quotation signs that needs to be cleaned in this data.

In [0]:
df_bronze.filter(~col("event_name").rlike(r"\([A-Z]{3}\)$")).select("event_name").distinct().limit(20).display()

### Number of invalid rows in relation to d(days) is: 62 727

In [0]:
df_days = df_bronze.filter(col("athlete_performance").rlike("(?i)d"))
print(f"Number of rows that contains days (d): {df_days.count()}")

In [0]:
df_days.select("event_name", "event_distance_length", "athlete_performance").display()

### How many nulls are there in important columns for silver obt?

| nulls_performance | nulls_birth_year | nulls_gender | nulls_speed |
| :--- | :--- | :--- | :--- |
| 2 | 588 161 | 7 | 224 |

- Only 2 rows are missing the athlete performance
- Almost 600k null values for athletes Year of Birth.
- Only 7 nulls on athlete gender
- Only 224 rows with nulls on athletes avarage speed

In [0]:
df_bronze.select(
    _sum(col("athlete_performance").isNull().cast("int")).alias("nulls_performance"),
    _sum(col("athlete_year_of_birth").isNull().cast("int")).alias("nulls_birth_year"),
    _sum(col("athlete_gender").isNull().cast("int")).alias("nulls_gender"),
    _sum(col("athlete_average_speed").isNull().cast("int")).alias("nulls_speed")
).display()

### Find dates that does NOT follow the standard pattern which is DD.MM.YYYY
- over 7k(7340) rows display odd formatting on dates. Example sample is:

| event_dates |
| :--- |
| 24.04.-08.05.2021 |
| 15.09.-02.10.2021 |
| 04.-09.10.2021 |
| 29.-30.01.2022 |
| 13.-14.08.2022 |
| 28.-29.10.2022 |
| 10.-20.03.1997 |
| 16.-17.10.2021 |
| 30.09.-02.10.2021 |
| 29.-30.12.2021 |
| 31.05.-03.06.2022 |


In [0]:
print("Deviations in event_dates:")
(df_bronze.filter(~col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$")).select("event_dates").distinct().display())

### Attempt to salvage all 7356 event_date rows and convert them to DateTypes
- with sparks import of `lit` i should be able to use sparks `right()` function I should be able to clean and convert all odd dates to DateTypes.
    - By importing `lit` and wrapping my 10 integer with it and using `right()` i could convert it to a literal column and make sense of my messy `event_dates` rows.


- The results of my attempt is:


| event_dates | clean_date_string | official_event_date | 
| :--- | :--- | :--- |
| 29.-30.05.2021 | 30.05.2021 | 2021-05-30 |
| 16.-18.04.2021 | 18.04.2021 | 2021-04-18 | 
| 30.04.-01.05.2021 | 01.05.2021 | 2021-05-01 |

    

In [0]:
# Pick out the last 10 chars from its string that should give me an end date
df_dates_test = df_bronze.withColumn("clean_date_string", right(col("event_dates"), lit(10)))

# Converting that string to a real date YYYY-MM-DD
df_dates_test = df_dates_test.withColumn("official_event_date", to_date(col("clean_date_string"), "dd.MM.yyyy"))


# Show the result (filtering on strings that contains - to see anomalies in data)
print("Conversion of messy dates:")
(df_dates_test
 .filter(col("event_dates").contains("-"))
 .select("event_dates", "clean_date_string", "official_event_date")
 .display())

### Finding anomalies in event_distance_length
- Trying to find outliers in these columns that arent `km`, `mi` or `h` that my pipeline will miss if I dont know about them.


- Rows containing odd distance values:
    - 108 292 rows

Example on how those rows look:

| event_distance_lenght | count | 
| :--- | :--- |
| 250km/6Etappen | 9192 |
| 6d | 9072 |
| 100km/2Etappen | 3817 |
| 250.7km/6Etappen | 2576 |
| 237km/6Etappen | 2309 |


In [0]:
print("====== ODD DISTANCE VALUES ====== ")
# hitta alla distancer som _INTE_ slutar på km, mi eller h (CASE INSENSITIVE)
# $ betyder 'i slutet av str'
odd_distances = df_bronze.filter(~col("event_distance_length").rlike("(?i)(km|mi|h)$"))

print(f"Nr of rows with anomalies in distances: {odd_distances.count()}")
odd_distances.groupBy("event_distance_length").count().orderBy(col("count").desc()).display()

### Finding anomalies in athlete_performance column
- Trying to find outliers in these columns that arent `km`, `mi` or `h` that my pipeline will miss if I dont know about them.

- Result from running a check on anomalies in `athlete_performance`-column when looking for more outliers in data after filtering away `nulls` and `d` in `athlete_performance` column gives:
    - `No rows returned` which is suspicious but completely plausible.
    

In [0]:
print("====== ODD PERFORMANCES ====== ")
# Performances bör normalt sett sluta på " h" eller " km".
# filtrerar bort nulls och "dagar" först då jag redan vet om nulls och 'd' per uppgiften
odd_performances = (
    df_bronze.filter(col("athlete_performance").isNotNull())
    .filter(~col("athlete_performance").rlike("(?i)d"))
    .filter(
        ~col("athlete_performance").rlike("(?i)( h| km| mi)$")
    )  # Letar efter det som INTE slutar på h/km/mi
)

print(f"Nr of rows with odd performances: {odd_performances.count()}")
odd_performances.select("event_distance_length", "athlete_performance").display()

### Gender distribution.
- Preliminary EDA for bronze layer displayed `nulls` in `athlete_gender`, only 7 of them but I still haven't found out what unique values there are in that column.
    - Are there spelling mistakes?
    - Are there versions of male/Male/MALE/m or female/Female/FEMALE/f

- The distribution of genders are:

| event_distance_lenght | count |
| :--- | :--- |
| M | 6 035 358 |
| F | 1 425 784 |
| X | 46 |
| `null` | 7 |




In [0]:
%python
print("Athlete gender distribution:")
df_bronze.groupBy("athlete_gender").count().orderBy(col("count").desc()).display()

### In first bronze EDA I noticed anomalies in `athlete_average_speed` could be extremly high (9999.0)
- Running at 9999km/h is faster than a fighterjet, this is clearly a mistake. 
    - 9999 COULD be a placeholder representing `Error` in some legacy messuring system.

- 1326 rows in `athlete_average_speed` column that does NOT look like a normal number related to speed.


- Speed anomalies in cell 26. This must be some big error. For example:

| event_name | athlete_average_speed | athlete_performance | speed_avg |
| :--- | :--- | :--- | :--- |
| South London Harriers 30 Mile (GBR) | 16764.0| 2:52:48 h | 16764 | 
| Guelph to Hamilton (CAN) | 16588.0 | 2:48:49 h | 16588 |
| 30mi track race Hong Kong (HKG) | 16577.0 | 2:54:45 h | 16577 | 
| South London Harriers 30 Mile (GBR) | 16455.0 | 2:56:03 h | 16455 | 
| South London Harriers 30 Mile (GBR) | 16311.0 | 2:57:36 h | 16311 | 
| South London Harriers 30 Mile (GBR) | 16297.0 | 2:57:45 h | 16297 | 
| South London Harriers 30 Mile (GBR) | 16282.0 | 2:57:55 h | 16282 | 
| Guelph to Hamilton (CAN) | 16281.0 | 2:52:00 h | 16281 | 
| South London Harriers 30 Mile (GBR) | 16221.0 | 2:58:35 h | 16221 | 
| South London Harriers 30 Mile (GBR) | 16171.0 | 2:59:08 h | 16171 | 

---

- Anomaly found. An average speed of 16764 km/h is not possible. There must be some kind of math error here.
- Lets take `South London Harriers 30 Mile (GBR) | 16764.0| 2:52:48 h | 16764 |` as an example: 
    - Distance: 30 miles = 48,28 km.
    - Time: 2 hours and 52 minutes. About 2,88 hours
    - Average speed: 48,28 / 2,88 = 16,76388888 km/h
    - Rounding it up it will be: 48,28 / 2,88 = 16,764 km/h. This must be the anomaly.

- When exporting this data from different measuring systems the data list one of its `,` signs.
    - `16764.0` really MUST be `16.764 km/h`. The number has been multiplies with 1000.
    - This means: ANYWHERE i find a speed that is OVER 1k km/h I must divide it by 1000 to get the real average speed out.



In [0]:
%python
print("Garbage in athlete_average_speed:")
(
    df_bronze.filter(col("athlete_average_speed").isNotNull())
    .filter(~trim(col("athlete_average_speed")).rlike(r"^\d+(\.\d+)?$"))
    .select("athlete_average_speed")
    .distinct()
    .display()
)

In [0]:
%python

print(
    "Athletes faster than Usain Bolt (Over 30km/h avarage speed on a long distance race):"
)

(df_bronze.withColumn("speed_avg", expr("try_cast(athlete_average_speed as double)"))
 .filter(col("speed_avg") > 30.0)
 .select("event_name", "athlete_average_speed", "athlete_performance", "speed_avg")
.display())

## Hard rules I must abide by after digging deeper into different races and world record holders for ultra marathons

The `athlete_average_speed` column is a column containing tons of anomalies. It mixes real speeds with lost decimals and lap times.

Since my primary goal as a Data Engineer is data validity, I can't try to patch this up with endless when() statements (like dividing by 1000 sometimes, but not always depending on what kind of race it is. This is not feasible)

Me as a Data Engineer will instead implement a solid sanity check:
1) The world record for 50 km (the shortest ultramarathon) is at an average speed of around 18.5 km/h. No human runs an ultramarathon faster than 25 km/h.

2) Instead of the previous / 1000 rule, I will update the OBT silver script with a logical rule: **Anything above 25 km/h** is obvious garbage from the source system and will be set to `Null`.

---
### Sources form a quick google search regarding ultramarathons:
---
**50 Kilometers (31.07 miles)**  
- **Men:** 2h 38m 43s. Set by CJ Albertson (USA) in 2022, averaging a blistering pace of 3:09 min/km (19.04 km/h).  
- **Women:** 2h59min 54s. Set by Des Linden (USA) in 2021, averaging 3:35 min/km (16.67 km/h)  
--- 
**100 Kliometers (62.14 miles)**  
- **Men (Unofficial):** 5h 59m 20s. Set by Sibusiso Kubheka (South Africa) in 2025. He is the first person to break the 6-hour mark, averaging a pace of ~3:35 min/km (16.7 km/h)  
- **Men (World Athletics Record):** 6h 05m 35s. Set by Aleksandr Sorokin (Lithuania) in 2023, averaging 3:39min/km (16.41 km/h)  
- **Women:** 6h 33m 11s. Set by a Japanese runner, averaging 3:56 min/km (15.26 km/h)  
--- 

**24 Hours:**  
- **Men:** 318.614 km (198.598 miles). Set by Aleksandr Sorokin (Lithuania) in 2022, sustaining an average speed of 13.31 km/h for the full day.  
- **Women:** 278.622 km (173.127 miles). Set by Sarah Webster (GBR) in 2025, maintaining an average speed of 11.60 km/h.  
